# CEG-WM：固定跨通道互谱读法，先零位移门槛

**复用现有图像，不重新生成。先做零位移密钥区分；不通过即停止，不启动位移对照。**

沿用成功的 **Python 3.12 GPU 环境**与 Secrets `HF_TOKEN`、`CEG_WM_ROOT_KEY`，运行全部单元。
只读输入：`MyDrive/CEG-WM/crop-bottleneck-v1/attempt01/`。
本轮独立输出：`MyDrive/CEG-WM/crop-cross-spectrum-v1/attempt01/`。

只加载原版 VAE。B 为原共同掩膜的实部 HF 相关；C 为固定跨通道互谱，全部 120 个通道对等权。
每个频率除以全部 16 通道总能量；整体做复数余弦的实部，不取绝对值。不选频率、不选通道、不选几何、不训练。
只用原密钥与固定 16 个错钥，不继承联合检测阈值、不混入 LF。

本地工程验证不是 GPU 科学结果。本 notebook 尚未在你的保存图像上执行；运行后可能正确地停在零位移门槛。


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, sys, json, re, time
SOURCE = pathlib.Path('/content/drive/MyDrive/CEG-WM/crop-bottleneck-v1/attempt01')
RUN_ID = 'attempt01'
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID)
output = pathlib.Path('/content/drive/MyDrive/CEG-WM/crop-cross-spectrum-v1') / RUN_ID
checkout = pathlib.Path('/content/cegwm-crop-cross-spectrum-source')
script_dir = pathlib.Path('/content/cegwm-crop-cross-spectrum')
script_dir.mkdir(exist_ok=True)
script = script_dir / 'run_cross_spectrum.py'
expected = 'd9ef4adb4f8424f1102f577bb76347fbba112bd9'
saved_env = json.loads((SOURCE / 'environment.json').read_text())
saved_protocol = json.loads((SOURCE / 'protocol.json').read_text())
saved_summary = json.loads((SOURCE / 'summary.json').read_text())
if saved_summary['status'] != 'COMPLETE_DIAGNOSTIC':
    raise RuntimeError('Original crop run is incomplete. No generation will be started.')
source_python = tuple(map(int, saved_env['python'].split('.')[:2]))
if sys.version_info[:2] != source_python:
    raise RuntimeError(f'Use the successful historical Colab Python {source_python[0]}.{source_python[1]} runtime; current Python is {sys.version.split()[0]}. No dependencies or source results were changed.')
child_env = dict(os.environ)
for name in ('HF_TOKEN', 'CEG_WM_ROOT_KEY'):
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f'Missing secret: {name}')
    child_env[name] = value
child_env['HF_HOME'] = '/content/cegwm-crop-v1-hf'
child_env['HF_HUB_CACHE'] = child_env['HF_HOME'] + '/hub'
child_env['PYTHONUNBUFFERED'] = '1'
child_env.pop('HF_HUB_OFFLINE', None)
child_env.pop('TRANSFORMERS_OFFLINE', None)

def run_logged(command, label, **kwargs):
    # Explicit PIPE+print: Colab may hide output inherited by subprocess.run.
    log_dir = output / 'execution_logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    log = log_dir / f'{label}-{time.time_ns()}.log'
    with log.open('x', encoding='utf-8') as sink:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, **kwargs) as process:
            for line in process.stdout:
                for key in ('HF_TOKEN', 'CEG_WM_ROOT_KEY'):
                    line = line.replace(child_env[key], '[REDACTED]')
                print(line, end='', flush=True)
                sink.write(line)
                sink.flush()
            code = process.wait()
    if code:
        raise RuntimeError(f'{label} failed with exit code {code}. Full output: {log}')

run_logged([sys.executable, '-c',
    'import torch; assert torch.cuda.is_available(), "Select a GPU runtime"; '
    'print({"python":__import__("sys").version,"gpu":torch.cuda.get_device_name(0),"torch":torch.__version__})'], 'gpu-check')
print('Read-only source:', SOURCE)
print('New output:', output)


In [ ]:
# Match the successful source reader dependencies; preserve installed CUDA torch.
packages = saved_env['packages']
names = ('diffusers','transformers','accelerate','numpy','Pillow','huggingface_hub','safetensors','sentencepiece')
dependencies = [f'{name}=={packages[name]}' for name in names]
torch_versions = subprocess.run([sys.executable, '-c',
    'import importlib.metadata as m,json; print(json.dumps({p:m.version(p) for p in ("torch","torchvision")}))'],
    capture_output=True, text=True, check=True)
constraints = pathlib.Path('/content/cegwm-cross-spectrum-cuda-constraints.txt')
constraints.write_text(''.join(f'{name}=={ver}\n' for name,ver in json.loads(torch_versions.stdout).items()))
run_logged([sys.executable, '-m', 'pip', 'install', '--only-binary=sentencepiece',
            '-c', str(constraints), *dependencies], 'pip-install')
if not checkout.exists():
    run_logged(['git', 'clone', 'https://github.com/RICHAAARC/CEG-WM.git', str(checkout)], 'clone')
run_logged(['git', '-C', str(checkout), 'checkout', '--detach', expected], 'checkout')
child_env['PYTHONPATH'] = str(checkout / 'src') + os.pathsep + str(checkout)
run_logged([sys.executable, '-c',
    'from diffusers import AutoencoderKL; from diffusers.image_processor import VaeImageProcessor; '
    'from cegwm.runtime.blind_scoring_v2 import _lf,_hf; '
    'print("VAE and frozen scoring imports OK; no models loaded")'], 'imports', env=child_env)


In [ ]:
# Embedded fixed runner plus hash-bound support modules.
(script_dir / 'run_residual.py').write_text('"""Read-only saved-image diagnostic. No generator, denoiser, or SyncSeal inference."""\nfrom __future__ import annotations\nimport argparse\nimport csv\nfrom dataclasses import asdict\nimport hashlib\nfrom importlib.metadata import version\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport uuid\n\nEXACT = "d9ef4adb4f8424f1102f577bb76347fbba112bd9"\nMODEL = "stabilityai/stable-diffusion-3.5-medium"\nTAU = 1.5545224858704942\nROLES = ("positive", "negative")\nIDS = tuple(f"v2-formal_evaluation_pairs-{i:04d}" for i in range(1, 9))\nATOL, RTOL = 1e-5, 1e-4\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for part in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef write(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    with tmp.open("x", encoding="utf-8") as f:\n        json.dump(data, f, ensure_ascii=False, indent=2, allow_nan=False)\n        f.flush()\n        os.fsync(f.fileno())\n    os.replace(tmp, path)\n\n\ndef bind(path, data):\n    if Path(path).exists():\n        if read(path) != data:\n            raise RuntimeError(f"Resume identity mismatch: {Path(path).name}; do not mix runs")\n    else:\n        write(path, data)\n\n\ndef error_data(error):\n    trace = "".join(traceback.format_exception(error))\n    for name in ("HF_TOKEN", "CEG_WM_ROOT_KEY"):\n        if os.environ.get(name):\n            trace = trace.replace(os.environ[name], "[REDACTED]")\n    return {"error_type": type(error).__name__, "traceback_redacted": trace}\n\n\ndef cases():\n    result = [{"id": "oracle_replay", "kind": "oracle"},\n              {"id": "predicted_replay", "kind": "predicted"},\n              {"id": "oracle_uncorrected_pixel_convention", "kind": "convention"},\n              {"id": "half_prediction_residual", "kind": "half_residual"}]\n    for axis in ("x", "y"):\n        for value in (-4., -2., -.5, .5, 2., 4.):\n            result.append({"id": f"translation_{axis}_{value:+g}px", "kind": "translation",\n                           "axis": axis, "value": value})\n    for value in (-.04, -.02, -.005, .005, .02, .04):\n        result.append({"id": f"scale_{value * 100:+g}pct", "kind": "scale", "value": value})\n    return result\n\n\ndef normalization():\n    import numpy as np\n    return np.array([[2 / 511, 0., -1.], [0., 2 / 511, -1.], [0., 0., 1.]])\n\n\ndef matrix_for(case, crop, predicted):\n    """All perturbations compose BEFORE one sampling from saved attacked RGB."""\n    import numpy as np\n    base = np.asarray(crop["sampler_h_observed_to_canonical"], dtype=float)\n    kind = case["kind"]\n    if kind == "oracle":\n        return base\n    if kind == "predicted":\n        return np.asarray(predicted, dtype=float)\n    if kind == "convention":\n        return np.asarray(crop["physical_h_observed_to_canonical"], dtype=float)\n    if kind == "half_residual":\n        # A single predeclared diagnostic point along the actual predicted-H error.\n        # No interpolation weight is selected using detection scores.\n        p = np.asarray(predicted, dtype=float)\n        return .5 * base / base[2, 2] + .5 * p / p[2, 2]\n    residual = np.eye(3)\n    if kind == "translation":\n        residual[0 if case["axis"] == "x" else 1, 2] = case["value"]\n    elif kind == "scale":\n        scale = 1 + case["value"]\n        residual[0, 0] = residual[1, 1] = scale\n        residual[0, 2] = residual[1, 2] = 256 * (1 - scale)\n    else:\n        raise ValueError("Unknown fixed diagnostic case")\n    n = normalization()\n    return n @ residual @ np.linalg.inv(n) @ base\n\n\ndef geometry_metrics(matrix, crop):\n    import numpy as np\n    n = normalization()\n    edge = np.linalg.inv(n) @ matrix @ n\n    plus = np.array([[1., 0., .5], [0., 1., .5], [0., 0., 1.]])\n    center = np.linalg.inv(plus) @ edge @ plus\n    actual = n @ center @ np.linalg.inv(n)\n    truth = np.asarray(crop["physical_h_observed_to_canonical"])\n    points = np.array([[-1., -1., 1.], [1., -1., 1.], [1., 1., 1.], [-1., 1., 1.], [0., 0., 1.]]).T\n    a, b = actual @ points, truth @ points\n    delta = (a[:2] / a[2] - b[:2] / b[2]) * 511 / 2\n    return {"effective_corner_rmse_px": float(np.sqrt(np.mean(np.sum(delta[:, :4] ** 2, axis=0)))),\n            "effective_center_shift_xy_px": delta[:, 4].tolist()}\n\n\ndef load_saved_image(source, artifact):\n    from PIL import Image\n    path = Path(artifact["path"]).resolve()\n    if not path.is_relative_to(source.resolve()):\n        raise RuntimeError("Saved image points outside SOURCE directory")\n    if sha(path) != artifact["sha256"]:\n        raise RuntimeError(f"Saved image hash differs: {path.name}")\n    with Image.open(path) as image:\n        if image.mode != "RGB" or image.size != (512, 512):\n            raise ValueError("Saved input must be RGB 512x512")\n        return image.copy()\n\n\ndef inspect_source(source):\n    protocol, summary = read(source / "protocol.json"), read(source / "summary.json")\n    if summary["status"] != "COMPLETE_DIAGNOSTIC" or protocol["source_exact"] != EXACT:\n        raise ValueError("Completed original crop diagnostic required")\n    if tuple(u["unit_id"] for u in protocol["roster"]) != IDS or protocol["reference_tau"] != TAU:\n        raise ValueError("Source roster or threshold differs")\n    indexed = {(r["unit_id"], r["role"], r["route"]): r for r in summary["records"]}\n    if len(indexed) != 64 or len(summary["records"]) != 64:\n        raise ValueError("Expected 64 unique source route records")\n    for uid in IDS:\n        for role in ROLES:\n            for route in ("clean", "crop_direct", "oracle_h", "predicted_h"):\n                row = indexed[uid, role, route]\n                if row["status"] != "SCORED":\n                    raise ValueError("This fixed diagnostic requires all saved baseline routes")\n                load_saved_image(source, row["image"])\n    return protocol, indexed\n\n\nclass Reader:\n    """Only the VAE plus the unchanged LF/HF arithmetic from the pinned source."""\n    def __init__(self, repo, source):\n        import torch\n        from diffusers import AutoencoderKL\n        from diffusers.image_processor import VaeImageProcessor\n        from cegwm.method.lf import (FrozenLFPublicAssets, LF_BALANCED_BLOCKS_CARRIER_METHOD_ID,\n            LF_BLOCKNORM_DETECTOR_STATISTIC_ID, LF_BALANCED_BLOCKS_EVALUATED_CANDIDATE_ID)\n        from cegwm.method.hf import FrozenHFPublicAssets\n        from cegwm.method.content_whitening import FrozenContentWhiteningLFPublicAssets\n        from experiments.run_blind_detection_v1 import load_whitening_asset_semantic, load_weighted_asset_semantic\n        from cegwm.shared.keys import normalize_detection_key\n        from cegwm.runtime.content_weighted_joint_sd35 import derive_stability_wrong_keys\n        revision = read(source / "model_revisions.json")[MODEL]\n        self.vae = AutoencoderKL.from_pretrained(MODEL, subfolder="vae", revision=revision,\n            torch_dtype=torch.float16, token=os.environ["HF_TOKEN"]).to("cuda").eval()\n        # Exactly StableDiffusion3Pipeline v0.39.0\'s image_processor construction.\n        self.processor = VaeImageProcessor(vae_scale_factor=2 ** (len(self.vae.config.block_out_channels) - 1))\n        lf = FrozenLFPublicAssets(self.vae, self.processor, MODEL + ":image_processor",\n            LF_BALANCED_BLOCKS_CARRIER_METHOD_ID, LF_BLOCKNORM_DETECTOR_STATISTIC_ID,\n            LF_BALANCED_BLOCKS_EVALUATED_CANDIDATE_ID)\n        self.lf = FrozenContentWhiteningLFPublicAssets(lf, load_whitening_asset_semantic(repo))\n        self.hf = FrozenHFPublicAssets(self.vae, self.processor, MODEL + ":image_processor")\n        self.weighted = load_weighted_asset_semantic(repo)\n        self.key = normalize_detection_key(os.environ["CEG_WM_ROOT_KEY"])\n        self.keys = (self.key, *derive_stability_wrong_keys(self.key))\n\n    def observe(self, image):\n        from cegwm.runtime.observation import encode_final_rgb_image\n        return encode_final_rgb_image(image, self.processor, self.vae)\n\n    def score(self, observation):\n        from cegwm.runtime.blind_scoring_v2 import _lf, _hf\n        from cegwm.method.content_weighted_joint import weighted_joint_score\n        from cegwm.method.blind_detection import statistic_from_weighted_scores\n        labels = ("registered", *(f"wrong_{i:02d}" for i in range(16)))\n        pairs = [(_lf(observation, key, self.lf), _hf(observation, key, self.hf)) for key in self.keys]\n        branches = {"lf": {label: p[0] for label, p in zip(labels, pairs)},\n                    "hf": {label: p[1] for label, p in zip(labels, pairs)},\n                    "weighted_joint": {label: weighted_joint_score(*p, self.weighted) for label, p in zip(labels, pairs)}}\n        stat = statistic_from_weighted_scores(branches["weighted_joint"])\n        return {"branches": branches, "statistic": asdict(stat), "margin": stat.value - TAU,\n                "above_reference_tau": stat.value > TAU}\n\n\ndef replay_comparison(current, old):\n    import numpy as np\n    diffs, passed = {}, True\n    for branch in ("lf", "hf", "weighted_joint"):\n        a = list(current["branches"][branch].values())\n        b = list(old["branches"][branch].values())\n        diffs[branch] = float(np.max(np.abs(np.asarray(a) - b)))\n        passed &= bool(np.allclose(a, b, atol=ATOL, rtol=RTOL))\n    passed &= bool(np.isclose(current["statistic"]["value"], old["statistic"]["value"], atol=ATOL, rtol=RTOL))\n    return {"passed": passed, "max_abs_branch_differences": diffs,\n            "statistic_difference": current["statistic"]["value"] - old["statistic"]["value"],\n            "atol": ATOL, "rtol": RTOL}\n\n\ndef observation_metrics(image, oracle_image, observation, oracle_observation):\n    import numpy as np\n    a, b = np.asarray(image, dtype=float) / 255, np.asarray(oracle_image, dtype=float) / 255\n    z, z0 = observation.detach().float().cpu().numpy(), oracle_observation.detach().float().cpu().numpy()\n    def relative(x, y):\n        return float(np.linalg.norm(x - y) / max(float(np.linalg.norm(y)), 1e-12))\n    return {"rgb_rmse_full": float(np.sqrt(np.mean((a-b)**2))),\n            "rgb_rmse_fixed_inner": float(np.sqrt(np.mean((a[64:448,64:448]-b[64:448,64:448])**2))),\n            "vae_relative_l2_full": relative(z, z0),\n            "vae_relative_l2_fixed_inner": relative(z[...,8:56,8:56], z0[...,8:56,8:56])}\n\n\ndef summarize(output):\n    rows, counts = [], {}\n    for uid in IDS:\n        for role in ROLES:\n            for case in cases():\n                path = output / "records" / uid / role / (case["id"] + ".json")\n                row = read(path) if path.exists() else {"status": "PENDING"}\n                rows.append({**row, "unit_id": uid, "role": role, "case": case["id"]})\n    for role in ROLES:\n        for case in cases():\n            subset = [r for r in rows if r["role"] == role and r["case"] == case["id"]]\n            scored = [r for r in subset if r["status"] == "SCORED"]\n            counts[f"{role}/{case[\'id\']}"] = {"planned": 8, "scored": len(scored),\n                "above_reference_tau": sum(r.get("above_reference_tau", False) for r in scored),\n                "operational_error": sum(r["status"] == "OPERATIONAL_ERROR" for r in subset),\n                "pending": sum(r["status"] == "PENDING" for r in subset),\n                "mean_margin": sum(r["margin"] for r in scored) / len(scored) if scored else None}\n    result = {"status": "COMPLETE_DIAGNOSTIC" if all(r["status"] == "SCORED" for r in rows) else "INCOMPLETE",\n              "source_pairs": 8, "cases_per_image": len(cases()), "planned_scores": len(rows),\n              "claim_ceiling": "fixed_grid_sensitivity_diagnosis_not_blind_recovery_or_calibrated_FPR",\n              "counts": counts, "records": rows}\n    write(output / "summary.json", result)\n    cols = ["unit_id", "role", "case", "status", "margin", "delta_score_vs_oracle",\n            "delta_registered_vs_oracle", "delta_wrongmax_vs_oracle", "delta_hf_vs_oracle",\n            "delta_lf_vs_oracle", "oracle_above_reference_tau", "lost_vs_oracle", "effective_corner_rmse_px",\n            "rgb_rmse_full", "rgb_rmse_fixed_inner", "vae_relative_l2_full", "vae_relative_l2_fixed_inner"]\n    tmp = output / "readout.csv.tmp"\n    with tmp.open("w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=cols)\n        writer.writeheader()\n        writer.writerows({k: r.get(k) for k in cols} for r in rows)\n    os.replace(tmp, output / "readout.csv")\n    return result\n\n\ndef run(source, output, repo):\n    import numpy as np\n    import torch\n    from cegwm.geometry_v7.r1b import rectify_attacked_rgb\n    from cegwm.shared.keys import normalize_detection_key, public_key_digest\n    if output.resolve().is_relative_to(source.resolve()) or source.resolve().is_relative_to(output.resolve()):\n        raise ValueError("SOURCE and OUTPUT must be separate non-overlapping directories")\n    protocol, old = inspect_source(source)  # Read-only verification BEFORE any inference.\n    if subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip() != EXACT:\n        raise ValueError("Historical source revision differs")\n    if subprocess.check_output(["git", "-C", str(repo), "diff", "HEAD", "--"], text=True):\n        raise ValueError("Tracked historical source has modifications")\n    if public_key_digest(normalize_detection_key(os.environ["CEG_WM_ROOT_KEY"])) != protocol["key_digest"]:\n        raise ValueError("Detection key differs from saved run")\n    identity = {"source": str(source.resolve()), "source_protocol_sha": sha(source / "protocol.json"),\n                "source_summary_sha": sha(source / "summary.json"), "source_model_revisions_sha": sha(source / "model_revisions.json"),\n                "source_environment_sha": sha(source / "environment.json"), "script_sha": sha(__file__),\n                "code_exact": EXACT, "cases": cases(), "tau_reference_only": TAU, "atol": ATOL, "rtol": RTOL}\n    bind(output / "protocol.json", identity)\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA VAE reader required; no image generation fallback")\n    env = {"python": platform.python_version(), "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,\n           "packages": {p: version(p) for p in ("torch", "torchvision", "diffusers", "transformers", "numpy", "Pillow", "huggingface_hub", "safetensors")}}\n    bind(output / "environment.json", env)\n    if summarize(output)["status"] == "COMPLETE_DIAGNOSTIC":\n        print("Already complete; no scoring repeated.", flush=True)\n        return\n    reader = Reader(repo, source)\n    crop = protocol["crop"]\n    references = {}\n\n    def process(uid, role, case, attacked, oracle_image, oracle_z, oracle_score):\n        path = output / "records" / uid / role / (case["id"] + ".json")\n        if path.exists() and read(path)["status"] == "SCORED":\n            return read(path)\n        attempt = output / "attempts" / uid / role / case["id"] / (uuid.uuid4().hex + ".json")\n        write(attempt, {"status": "STARTED", "time": time.time()})\n        try:\n            predicted = old[uid, role, "predicted_h"]["h_passed_to_frozen_renderer"]\n            matrix = matrix_for(case, crop, predicted)\n            image = rectify_attacked_rgb(attacked, matrix.tolist())\n            if case["kind"] in ("oracle", "predicted"):\n                previous = old[uid, role, "oracle_h" if case["kind"] == "oracle" else "predicted_h"]\n                if not np.array_equal(np.asarray(image), np.asarray(load_saved_image(source, previous["image"]))):\n                    raise RuntimeError("Replay RGB differs from saved route; stop before residual tests")\n            z = oracle_z if case["kind"] == "oracle" else reader.observe(image)\n            scored = reader.score(z)\n            replay = replay_comparison(scored, previous) if case["kind"] in ("oracle", "predicted") else None\n            if replay is not None and not replay["passed"]:\n                write(attempt, {"status": "REPLAY_MISMATCH", "replay": replay, **scored})\n                raise RuntimeError("VAE-only reader replay mismatch; residual diagnosis halted")\n            a, b = scored["statistic"], oracle_score["statistic"]\n            result = {"status": "SCORED", "h_passed_to_renderer": matrix.tolist(), **scored,\n                      **geometry_metrics(matrix, crop), **observation_metrics(image, oracle_image, z, oracle_z),\n                      "replay": replay, "delta_score_vs_oracle": a["value"] - b["value"],\n                      "delta_registered_vs_oracle": a["registered_weighted_joint"] - b["registered_weighted_joint"],\n                      "delta_wrongmax_vs_oracle": a["wrong_key_max"] - b["wrong_key_max"],\n                      "delta_hf_vs_oracle": scored["branches"]["hf"]["registered"] - oracle_score["branches"]["hf"]["registered"],\n                      "delta_lf_vs_oracle": scored["branches"]["lf"]["registered"] - oracle_score["branches"]["lf"]["registered"],\n                      "oracle_above_reference_tau": oracle_score["above_reference_tau"],\n                      "lost_vs_oracle": oracle_score["above_reference_tau"] and not scored["above_reference_tau"]}\n            write(attempt, result)\n            write(path, result)\n            print(f"{uid[-4:]} {role} {case[\'id\']}: margin={scored[\'margin\']:.4f}", flush=True)\n            return result\n        except Exception as error:\n            result = {"status": "OPERATIONAL_ERROR", **error_data(error)}\n            # Keep detailed replay values if available, alongside the final error.\n            if attempt.exists() and read(attempt).get("status") == "REPLAY_MISMATCH":\n                result["replay_failure"] = read(attempt)\n            write(attempt, result)\n            write(path, result)\n            summarize(output)\n            raise\n\n    # Complete ALL 32 replay controls before starting any perturbation inference.\n    for uid in IDS:\n        for role in ROLES:\n            attacked = load_saved_image(source, old[uid, role, "crop_direct"]["image"])\n            oracle_image = load_saved_image(source, old[uid, role, "oracle_h"]["image"])\n            oracle_z = reader.observe(oracle_image)\n            oracle_score = reader.score(oracle_z)\n            check = replay_comparison(oracle_score, old[uid, role, "oracle_h"])\n            if not check["passed"]:\n                write(output / "replay_gate.json", {"status": "FAILED", "unit_id": uid, "role": role, "check": check})\n                raise RuntimeError("Oracle reader replay mismatch; no residual tests started")\n            references[uid, role] = (attacked, oracle_image, oracle_z, oracle_score)\n            for case in cases()[:2]:\n                process(uid, role, case, *references[uid, role])\n            summarize(output)\n    write(output / "replay_gate.json", {"status": "PASSED", "planned": 32, "atol": ATOL, "rtol": RTOL})\n    for uid in IDS:\n        for role in ROLES:\n            for case in cases()[2:]:\n                process(uid, role, case, *references[uid, role])\n            summarize(output)\n    result = summarize(output)\n    print(json.dumps({"status": result["status"], "counts": result["counts"]}, indent=2), flush=True)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--source", type=Path, required=True)\n    p.add_argument("--output", type=Path, required=True)\n    p.add_argument("--repo", type=Path, required=True)\n    args = p.parse_args()\n    # Guard this even for error reporting: never create a file inside SOURCE.\n    if args.output.resolve().is_relative_to(args.source.resolve()) or args.source.resolve().is_relative_to(args.output.resolve()):\n        raise ValueError("SOURCE and OUTPUT must not overlap")\n    sys.path[:0] = [str(args.repo), str(args.repo / "src")]\n    try:\n        run(args.source, args.output, args.repo)\n    except Exception as error:\n        write(args.output / "last_failure.json", error_data(error))\n        raise\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(script_dir / 'run_mechanism.py').write_text('"""Saved-RGB mechanism probes. Counterfactual latent scores are NOT blind decisions."""\nfrom __future__ import annotations\nimport argparse\nimport csv\nfrom importlib.metadata import version\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport sys\nimport time\nimport uuid\n\n# Same validated VAE-only reader and read-only source verifier as the previous run.\nimport run_residual as support\nfrom run_residual import (Reader, EXACT, MODEL, IDS, ROLES, ATOL, RTOL, read, write,\n    sha, bind, error_data, inspect_source, load_saved_image, replay_comparison)\n\nSUPPORT_SHA = "27945052b071b4a207d674c60cfd79f8d326448a09f5bdb3f54524cc31c1c701"\nSHIFTS = ((0,0),(-2,0),(2,0),(0,-2),(0,2),(8,0),(0,8))\n\n\ndef shift_id(dx,dy):\n    return f"dx{dx:+d}_dy{dy:+d}"\n\n\ndef rgb_shift(image, dx, dy, periodic=False):\n    """Exact uint8 permutation/copy. No RGB interpolation or PIL resampling."""\n    import numpy as np\n    from PIL import Image\n    arr=np.asarray(image)\n    shifted=np.roll(arr,(dy,dx),axis=(0,1)).copy()\n    if not periodic:\n        if dx>0: shifted[:,:dx]=0\n        if dx<0: shifted[:,dx:]=0\n        if dy>0: shifted[:dy]=0\n        if dy<0: shifted[dy:]=0\n    return Image.fromarray(shifted)\n\n\ndef project(z):\n    """Common real subspace without even-grid Nyquist row/column."""\n    import torch\n    f=torch.fft.fft2(z.float(),norm="ortho")\n    h,w=z.shape[-2:]\n    if h%2==0: f[...,h//2,:]=0\n    if w%2==0: f[...,:,w//2]=0\n    return torch.fft.ifft2(f,norm="ortho").real\n\n\ndef phase_shift(z, dx, dy):\n    """Unitary periodic Fourier translation on odd grids or the common subspace."""\n    import torch\n    h,w=z.shape[-2:]\n    f=torch.fft.fft2(z.float(),norm="ortho")\n    fy=torch.fft.fftfreq(h,device=z.device)[:,None]\n    fx=torch.fft.fftfreq(w,device=z.device)[None,:]\n    phase=torch.exp((-2j*math.pi)*(fx*dx+fy*dy))\n    shifted=torch.fft.ifft2(f*phase,norm="ortho")\n    # Do not silently throw away imaginary Nyquist energy on even grids.\n    if float(shifted.imag.norm()) > 2e-5*max(float(shifted.real.norm()),1e-12):\n        raise ValueError("Fourier shift left the real common subspace")\n    return shifted.real\n\n\ndef padded_phase(z, dx, dy, pad=32):\n    import torch.nn.functional as F\n    # 64+2*pad+1 is odd: no ambiguous Nyquist mode in the extension grid.\n    h,w=z.shape[-2:]\n    extended=F.pad(z,(pad,pad+1,pad,pad+1),mode="constant",value=0.)\n    moved=phase_shift(extended,dx,dy)\n    return moved[...,pad:pad+h,pad:pad+w]\n\n\ndef integer_roll(z, dx, dy, periodic):\n    import torch\n    result=torch.roll(z,(dy,dx),dims=(-2,-1)).clone()\n    if not periodic:\n        if dx>0: result[...,:,:dx]=0\n        if dx<0: result[...,:,dx:]=0\n        if dy>0: result[...,:dy,:]=0\n        if dy<0: result[...,dy:,:]=0\n    return result\n\n\ndef bilinear_shift(z, dx, dy, periodic):\n    ix,iy=math.floor(dx),math.floor(dy)\n    ax,ay=dx-ix,dy-iy\n    return sum(weight*integer_roll(z,ix+i,iy+j,periodic)\n               for i,j,weight in ((0,0,(1-ax)*(1-ay)),(1,0,ax*(1-ay)),\n                                  (0,1,(1-ax)*ay),(1,1,ax*ay)) if weight)\n\n\ndef relative(a,b):\n    return float((a-b).float().norm()/b.float().norm().clamp_min(1e-12))\n\n\ndef remove_fraction(z):\n    import torch\n    f=torch.fft.fft2(z.float(),norm="ortho")\n    removed=f.clone()\n    removed[...,z.shape[-2]//2,:]=0\n    removed[...,:,z.shape[-1]//2]=0\n    return float(((f-removed).abs()**2).sum()/(f.abs()**2).sum().clamp_min(1e-20))\n\n\nclass HFProbe:\n    """Original real-spectrum normalized cosine, full and fixed non-Nyquist masks."""\n    def __init__(self, reader, shape, device):\n        import torch\n        from cegwm.method.hf import _spec\n        from cegwm.method.frequency import radial_frequency_mask,reconstruct_frequency_carrier\n        spec=_spec(reader.hf)\n        h,w=shape[-2:]\n        full=torch.from_numpy(radial_frequency_mask(h,w,spec)).to(device)\n        common=full.clone()\n        common[h//2,:]=False\n        common[:,w//2]=False\n        self.masks={"full":full,"common":common}\n        self.templates={}\n        self.labels=("registered",*(f"wrong_{i:02d}" for i in range(16)))\n        spectra=[torch.fft.rfft2(reconstruct_frequency_carrier(key,shape,spec,dtype=torch.float32,\n                    device=device),norm="ortho").real for key in reader.keys]\n        for mode,mask in self.masks.items():\n            vectors=[]\n            for f in spectra:\n                v=f[...,mask].reshape(-1).double()\n                v=v-v.mean()\n                vectors.append(v/v.norm())\n            self.templates[mode]=torch.stack(vectors)\n\n    def score(self,z,mode="common"):\n        import torch\n        f=torch.fft.rfft2(z.float(),norm="ortho").real\n        v=f[...,self.masks[mode]].reshape(-1).double()\n        v=v-v.mean()\n        norm=v.norm()\n        if not torch.isfinite(norm) or float(norm)==0:\n            raise RuntimeError("Undefined HF diagnostic cosine")\n        values=(self.templates[mode]@(v/norm)).tolist()\n        if not all(math.isfinite(x) for x in values):\n            raise RuntimeError("Nonfinite diagnostic score")\n        return {"scores":dict(zip(self.labels,values)),"registered":values[0],\n                "wrong_max":max(values[1:]),"gap":values[0]-max(values[1:])}\n\n\ndef operator_fields(z0,black,z_actual,z_circular,dx,dy):\n    """Translate residual relative to encoded black; black baseline stays fixed."""\n    r=project(z0-black)\n    ax,ay=dx/8,dy/8\n    phase=phase_shift(r,ax,ay)\n    roundtrip=relative(phase_shift(phase,-ax,-ay),r)\n    if roundtrip>2e-5:\n        raise RuntimeError("Fourier phase round-trip failed")\n    actual=project(z_actual-black)\n    circular=project(z_circular-black)\n    fields={\n        "base":black+r,\n        "phase_periodic":black+phase,\n        "phase_pad32":black+padded_phase(r,ax,ay,32),\n        "phase_pad64":black+padded_phase(r,ax,ay,64),\n        "bilinear_periodic":black+bilinear_shift(r,ax,ay,True),\n        "bilinear_zero":black+bilinear_shift(r,ax,ay,False),\n        "actual_zero":black+actual,\n        "actual_circular":black+circular,\n        "actual_inverse_periodic":black+phase_shift(circular,-ax,-ay),\n        "actual_inverse_pad32":black+padded_phase(actual,-ax,-ay,32),\n        "phase_periodic_unsubtracted":phase_shift(project(z0),ax,ay),\n    }\n    expected=project(fields["phase_pad32"]-black)\n    metrics={"phase_roundtrip_relative_l2":roundtrip,\n             "phase_periodic_energy_ratio":float(phase.float().norm()/r.float().norm().clamp_min(1e-12)),\n             "nyquist_removed_energy_fraction_base_residual":remove_fraction(z0-black),\n             "nyquist_removed_energy_fraction_actual_residual":remove_fraction(z_actual-black),\n             "extra_relative_l2_full":relative(actual,expected),\n             "extra_relative_l2_inner":relative(actual[...,8:56,8:56],expected[...,8:56,8:56]),\n             "padding_convergence_relative_l2":relative(fields["phase_pad32"]-black,fields["phase_pad64"]-black)}\n    return fields,metrics\n\n\ndef decompose(scores):\n    """Signed telescoping score changes, NOT causal variance percentages."""\n    result={}\n    for metric in ("registered","gap"):\n        s={name:value[metric] for name,value in scores.items()}\n        result[metric]={\n            "total_loss":s["base"]-s["actual_zero"],\n            "phase_loss":s["base"]-s["phase_periodic"],\n            "boundary_model_delta":s["phase_periodic"]-s["phase_pad32"],\n            "extra_vs_padded_phase":s["phase_pad32"]-s["actual_zero"],\n            "bilinear_model_delta":s["phase_pad32"]-s["bilinear_zero"],\n            "extra_vs_bilinear":s["bilinear_zero"]-s["actual_zero"],\n            "periodic_interpolation_delta":s["phase_periodic"]-s["bilinear_periodic"],\n            "padding_32_vs_64":s["phase_pad32"]-s["phase_pad64"],\n            "rgb_boundary_delta":s["actual_circular"]-s["actual_zero"],\n            "black_baseline_model_delta":s["phase_periodic"]-s["phase_periodic_unsubtracted"],\n            "inverse_periodic_gain":s["actual_inverse_periodic"]-s["actual_circular"],\n            "inverse_pad32_gain":s["actual_inverse_pad32"]-s["actual_zero"],\n            "inverse_pad32_shortfall":s["base"]-s["actual_inverse_pad32"],\n        }\n    return result\n\n\ndef summarize(output):\n    rows=[]\n    for uid in IDS:\n        for role in ROLES:\n            for dx,dy in SHIFTS:\n                path=output/"records"/uid/role/(shift_id(dx,dy)+".json")\n                r=read(path) if path.exists() else {"status":"PENDING"}\n                rows.append({**r,"unit_id":uid,"role":role,"dx_rgb":dx,"dy_rgb":dy})\n    result={"status":"COMPLETE_DIAGNOSTIC" if all(r["status"]=="SCORED" for r in rows) else "INCOMPLETE",\n            "source_pairs":8,"planned_records":112,"records":rows,\n            "claim_ceiling":"oracle_translation_mechanism_probe_no_blind_detector_or_formal_FPR_claim"}\n    counts={}\n    for role in ROLES:\n        for dx,dy in SHIFTS:\n            rs=[r for r in rows if r["role"]==role and (r["dx_rgb"],r["dy_rgb"])==(dx,dy)]\n            ok=[r for r in rs if r["status"]=="SCORED"]\n            counts[role+"/"+shift_id(dx,dy)]={"planned":8,"scored":len(ok),\n                "pending":sum(r["status"]=="PENDING" for r in rs),\n                "operational_error":sum(r["status"]=="OPERATIONAL_ERROR" for r in rs),\n                "actual_production_above_reference_tau":sum(r["actual_production"]["above_reference_tau"] for r in ok)}\n    result["counts"]=counts\n    write(output/"summary.json",result)\n    flat=[]\n    for r in rows:\n        f={k:r[k] for k in ("unit_id","role","dx_rgb","dy_rgb","status")}\n        if r["status"]=="SCORED":\n            f.update(r["metrics"])\n            f["rgb_periodic_zero_identical"]=r["rgb_periodic_zero_identical"]\n            f["actual_production_margin"]=r["actual_production"]["margin"]\n            f["hf_full_actual"]=r["full_hf_actual"]["registered"]\n            f["hf_full_base"]=r["full_hf_base"]["registered"]\n            f["hf_common_actual"]=r["hf_diagnostic"]["actual_zero"]["registered"]\n            f["hf_common_base"]=r["hf_diagnostic"]["base"]["registered"]\n            f["hf_full_registered_loss"]=f["hf_full_base"]-f["hf_full_actual"]\n            f["hf_full_gap_loss"]=r["full_hf_base"]["gap"]-r["full_hf_actual"]["gap"]\n            for metric,changes in r["decomposition"].items():\n                f.update({metric+"_"+k:v for k,v in changes.items()})\n        flat.append(f)\n    columns=list(dict.fromkeys(k for r in flat for k in r))\n    tmp=output/"readout.csv.tmp"\n    with tmp.open("w",encoding="utf-8",newline="") as stream:\n        writer=csv.DictWriter(stream,fieldnames=columns)\n        writer.writeheader()\n        writer.writerows(flat)\n    os.replace(tmp,output/"readout.csv")\n    return result\n\n\ndef run(source,output,repo):\n    import numpy as np\n    import torch\n    from PIL import Image\n    from cegwm.shared.keys import normalize_detection_key,public_key_digest\n    if output.resolve().is_relative_to(source.resolve()) or source.resolve().is_relative_to(output.resolve()):\n        raise ValueError("SOURCE and OUTPUT must not overlap")\n    if sha(support.__file__)!=SUPPORT_SHA:\n        raise ValueError("Validated VAE-only support module identity differs")\n    protocol,old=inspect_source(source)\n    if subprocess.check_output(["git","-C",str(repo),"rev-parse","HEAD"],text=True).strip()!=EXACT:\n        raise ValueError("Historical source revision differs")\n    if subprocess.check_output(["git","-C",str(repo),"diff","HEAD","--"],text=True):\n        raise ValueError("Historical tracked source modified")\n    if public_key_digest(normalize_detection_key(os.environ["CEG_WM_ROOT_KEY"]))!=protocol["key_digest"]:\n        raise ValueError("Key differs from source run")\n    identity={"schema":"crop_mechanism_v1","source":str(source.resolve()),"script_sha":sha(__file__),\n              "support_sha":SUPPORT_SHA,"source_exact":EXACT,"shifts_rgb":[list(s) for s in SHIFTS],\n              "source_hashes":{name:sha(source/name) for name in ("protocol.json","summary.json","environment.json","model_revisions.json")},\n              "boundary":"encoded_black_residual_circular_vs_zero_extension",\n              "phase":"Nyquist_row_column_removed_common_HF_subspace",\n              "padding_latent":[32,64],"inner_latent":[8,56],"replay_atol":ATOL,"replay_rtol":RTOL}\n    bind(output/"protocol.json",identity)\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA VAE reader required; no generation fallback")\n    environment={"python":platform.python_version(),"gpu":torch.cuda.get_device_name(0),"cuda":torch.version.cuda,\n        "packages":{p:version(p) for p in ("torch","torchvision","diffusers","transformers","numpy","Pillow","huggingface_hub","safetensors")}}\n    bind(output/"environment.json",environment)\n    if summarize(output)["status"]=="COMPLETE_DIAGNOSTIC":\n        print("Already complete; no scoring repeated.",flush=True)\n        return\n    reader=Reader(repo,source)\n    black=reader.observe(Image.new("RGB",(512,512),(0,0,0))).float()\n    if tuple(black.shape)!=(1,16,64,64):\n        raise ValueError("Unexpected VAE observation shape")\n    probe=HFProbe(reader,tuple(black.shape),black.device)\n    refs={}\n    # All 16 saved Oracle baselines must replay before the first mechanism probe.\n    for uid in IDS:\n        for role in ROLES:\n            image=load_saved_image(source,old[uid,role,"oracle_h"]["image"])\n            z=reader.observe(image)\n            scored=reader.score(z)\n            check=replay_comparison(scored,old[uid,role,"oracle_h"])\n            full=probe.score(z,"full")\n            hf_ok=np.allclose(list(full["scores"].values()),list(scored["branches"]["hf"].values()),atol=ATOL,rtol=RTOL)\n            if not check["passed"] or not hf_ok:\n                write(output/"replay_gate.json",{"status":"FAILED","unit_id":uid,"role":role,"check":check,"cached_HF_matches":bool(hf_ok)})\n                raise RuntimeError("Baseline/HF scoring replay failed; no mechanism probes started")\n            refs[uid,role]=(image,z,scored)\n    write(output/"replay_gate.json",{"status":"PASSED","planned":16,"atol":ATOL,"rtol":RTOL})\n    for uid in IDS:\n        for role in ROLES:\n            image,z0,base_production=refs[uid,role]\n            for dx,dy in SHIFTS:\n                path=output/"records"/uid/role/(shift_id(dx,dy)+".json")\n                if path.exists() and read(path)["status"]=="SCORED": continue\n                attempt=output/"attempts"/uid/role/shift_id(dx,dy)/(uuid.uuid4().hex+".json")\n                write(attempt,{"status":"STARTED","time":time.time()})\n                try:\n                    zero=rgb_shift(image,dx,dy,False)\n                    circular=rgb_shift(image,dx,dy,True)\n                    same=bool(np.array_equal(np.asarray(zero),np.asarray(circular)))\n                    actual=z0 if (dx,dy)==(0,0) else reader.observe(zero)\n                    zcirc=actual if same else reader.observe(circular)\n                    production=base_production if (dx,dy)==(0,0) else reader.score(actual)\n                    fields,metrics=operator_fields(z0.float(),black,actual.float(),zcirc.float(),dx,dy)\n                    diagnostic={name:probe.score(z) for name,z in fields.items()}\n                    if (dx,dy)==(0,0):\n                        for name,z in fields.items():\n                            if relative(project(z),project(z0))>3e-5:\n                                raise RuntimeError(f"Zero-shift control differs: {name}")\n                    result={"status":"SCORED","rgb_shift_is_exact_integer":True,\n                            "rgb_periodic_zero_identical":same,"actual_production":production,\n                            "full_hf_actual":probe.score(actual,"full"),\n                            "full_hf_base":probe.score(z0,"full"),"hf_diagnostic":diagnostic,\n                            "metrics":metrics,"decomposition":decompose(diagnostic)}\n                    write(attempt,result)\n                    write(path,result)\n                    changes=result["decomposition"]["registered"]\n                    print(f"{uid[-4:]} {role} {shift_id(dx,dy)}: HF total={changes[\'total_loss\']:.5f}, phase={changes[\'phase_loss\']:.5f}, extra={changes[\'extra_vs_padded_phase\']:.5f}",flush=True)\n                except Exception as error:\n                    failed={"status":"OPERATIONAL_ERROR",**error_data(error)}\n                    write(attempt,failed);write(path,failed);summarize(output)\n                    raise\n            summarize(output)\n    final=summarize(output)\n    print(__import__("json").dumps({"status":final["status"],"counts":final["counts"]},indent=2),flush=True)\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    for name in ("source","output","repo"): parser.add_argument("--"+name,type=Path,required=True)\n    args=parser.parse_args()\n    if args.output.resolve().is_relative_to(args.source.resolve()) or args.source.resolve().is_relative_to(args.output.resolve()):\n        raise ValueError("SOURCE and OUTPUT must not overlap")\n    sys.path[:0]=[str(args.repo),str(args.repo/"src")]\n    try: run(args.source,args.output,args.repo)\n    except Exception as error:\n        write(args.output/"last_failure.json",error_data(error))\n        raise\n\n\nif __name__=="__main__": main()\n', encoding='utf-8')
(script_dir / 'run_cross_spectrum.py').write_text('"""One fixed cross-channel reader; a failed zero-shift gate forbids shift probes."""\nfrom __future__ import annotations\nimport argparse\nimport csv\nfrom importlib.metadata import version\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport statistics\nimport subprocess\nimport sys\nimport uuid\n\nimport run_residual as support\nimport run_mechanism as geometry\nfrom run_residual import (Reader, EXACT, IDS, ROLES, ATOL, RTOL, read, write, sha,\n                         bind, error_data, inspect_source, load_saved_image, replay_comparison)\n\nSUPPORT_SHA=\'27945052b071b4a207d674c60cfd79f8d326448a09f5bdb3f54524cc31c1c701\'\nGEOMETRY_SHA=\'9bc94ed7a467f2030bedb33bb915eba3969aceaa43db3e5b03458fabceb64f52\'\nSHIFTS=((0,0),(-2,0),(2,0),(0,-2),(0,2),(8,0),(0,8))\nSMALL=SHIFTS[1:5]\nLABELS=(\'registered\',*(f\'wrong_{j:02d}\' for j in range(16)))\nRULES={\'zero_top1_min\':7,\'zero_pair_min\':7,\'zero_margin_retention\':.8,\n       \'drift_ratio_max\':.7,\'improved_sources_min\':6,\'shift_top1_min\':6,\n       \'shift_pair_min\':6,\'shift_margin_retention\':.5,\'negative_top1_stop\':3,\n       \'phase_score_atol\':1e-6}\nTERMINAL={\'STOPPED_ZERO_GATE\',\'STOPPED_UNINFORMATIVE_REFERENCE\',\n          \'COMPLETE_CANDIDATE_PASSED\',\'COMPLETE_CANDIDATE_REJECTED\'}\n\n\ndef cross_features(z, mask):\n    import torch\n    z=z.detach().to(device=\'cpu\',dtype=torch.float64)\n    if tuple(z.shape)!=(1,16,64,64) or not torch.isfinite(z).all():\n        raise ValueError(\'Expected finite 1x16x64x64 observation\')\n    spectrum=torch.fft.rfft2(z,norm=\'ortho\')[0,:,mask]\n    energy=spectrum.abs().square().sum(dim=0)\n    c,d=torch.triu_indices(16,16,offset=1)\n    # Zero-energy frequencies contribute zero; no data-dependent mask or tuning.\n    denom=torch.where(energy>0,energy,torch.ones_like(energy))\n    return (spectrum[c]*spectrum[d].conj()/denom).reshape(-1)\n\n\ndef keyed(values):\n    values=[float(v) for v in values]\n    if len(values)!=17 or not all(math.isfinite(v) for v in values):\n        raise ValueError(\'Seventeen finite scores required\')\n    wrong=max(values[1:])\n    return {\'scores\':dict(zip(LABELS,values)), \'registered\':values[0],\n            \'wrong_max\':wrong,\'gap\':values[0]-wrong,\'top1\':values[0]>wrong}\n\n\nclass FixedProbe:\n    def __init__(self,reader):\n        import torch\n        from cegwm.method.hf import _spec\n        from cegwm.method.frequency import reconstruct_frequency_carrier\n        self.linear=geometry.HFProbe(reader,(1,16,64,64),\'cpu\')\n        self.mask=self.linear.masks[\'common\']\n        templates=[]\n        for key in reader.keys:\n            carrier=reconstruct_frequency_carrier(key,(1,16,64,64),_spec(reader.hf),\n                                                  dtype=torch.float32,device=\'cpu\')\n            vector=cross_features(carrier,self.mask)\n            norm=vector.norm()\n            if not torch.isfinite(norm) or float(norm)==0: raise ValueError(\'Undefined key template\')\n            templates.append(vector/norm)\n        self.templates=torch.stack(templates)\n        # Coincident relation templates would invalidate this fixed 17-key test.\n        for i in range(17):\n            for j in range(i):\n                if float((self.templates[i]-self.templates[j]).norm())<1e-10:\n                    raise ValueError(\'Relation-template collision in fixed keys\')\n\n    def score(self,z):\n        import torch\n        z=z.detach().cpu()\n        linear=self.linear.score(z,\'common\')\n        feature=cross_features(z,self.mask)\n        norm=feature.norm()\n        if not torch.isfinite(norm) or float(norm)==0:\n            raise ValueError(\'Undefined cross-spectrum score; cannot count a flat reader as stable\')\n        values=(self.templates.conj()@(feature/norm)).real.tolist()\n        return {\'B\':keyed(linear[\'scores\'].values()),\'C\':keyed(values)}\n\n\ndef robust_phase_shift(z,dx,dy):\n    """Double-precision pure phase control of Qz, no encoded-black subtraction."""\n    import torch\n    f=torch.fft.fft2(z.detach().cpu().double(),norm=\'ortho\')\n    f[...,32,:]=0\n    f[...,:,32]=0\n    freq=torch.fft.fftfreq(64,dtype=torch.float64)\n    phase=torch.exp(-2j*math.pi*(freq[None,:]*dx/8+freq[:,None]*dy/8))\n    moved=torch.fft.ifft2(f*phase,norm=\'ortho\')\n    if float(moved.imag.norm())>1e-10*max(float(moved.real.norm()),1e-12):\n        raise RuntimeError(\'Non-real pure-phase control\')\n    return moved.real\n\n\ndef scale_from_negatives(zero,method):\n    # Population within-image dispersion of the same 16 wrong keys, pooled over 8 negatives.\n    variances=[]\n    for uid in IDS:\n        scores=zero[uid,\'negative\'][\'readers\'][method][\'scores\']\n        wrong=[scores[label] for label in LABELS[1:]]\n        mean=statistics.mean(wrong)\n        variances.extend((v-mean)**2 for v in wrong)\n    value=math.sqrt(statistics.mean(variances))\n    if not math.isfinite(value) or value<=0: raise ValueError(\'Zero/nonfinite baseline wrong-key scale\')\n    return value\n\n\ndef assess_zero(zero):\n    scales={m:scale_from_negatives(zero,m) for m in (\'B\',\'C\')}\n    medians={m:statistics.median(zero[u,\'positive\'][\'readers\'][m][\'gap\']/scales[m] for u in IDS)\n             for m in (\'B\',\'C\')}\n    top1=sum(zero[u,\'positive\'][\'readers\'][\'C\'][\'top1\'] for u in IDS)\n    negative_top1=sum(zero[u,\'negative\'][\'readers\'][\'C\'][\'top1\'] for u in IDS)\n    pair=sum(zero[u,\'positive\'][\'readers\'][\'C\'][\'registered\']>\n             zero[u,\'negative\'][\'readers\'][\'C\'][\'registered\'] for u in IDS)\n    checks={\'positive_top1_7_of_8\':top1>=RULES[\'zero_top1_min\'],\n            \'paired_positive_gt_negative_7_of_8\':pair>=RULES[\'zero_pair_min\'],\n            \'positive_median_gap\':medians[\'C\']>0,\n            \'baseline_margin_retention_80pct\':medians[\'C\']>=RULES[\'zero_margin_retention\']*medians[\'B\'],\n            \'negative_top1_below_3_of_8\':negative_top1<RULES[\'negative_top1_stop\']}\n    reference_ok=medians[\'B\']>0\n    return {\'status\':\'PASSED\' if reference_ok and all(checks.values()) else\n            (\'FAILED\' if reference_ok else \'REFERENCE_UNINFORMATIVE\'),\n            \'checks\':checks,\'scales\':scales,\'median_standardized_gap\':medians,\n            \'positive_top1\':top1,\'paired_positive_gt_negative\':pair,\'negative_top1\':negative_top1,\n            \'denominator\':8,\'scale_definition\':\'sqrt(mean over 8 negative images x 16 wrong keys of within-image squared score deviations)\'}\n\n\ndef assess_shifts(records,zero_gate):\n    scales=zero_gate[\'scales\']\n    def score(uid,role,shift,method): return records[uid,role,*shift][\'readers\'][method]\n    drift={m:[max(abs(score(u,\'positive\',s,m)[\'gap\']-score(u,\'positive\',(0,0),m)[\'gap\'])/scales[m]\n                    for s in SMALL) for u in IDS] for m in (\'B\',\'C\')}\n    median={m:statistics.median(drift[m]) for m in (\'B\',\'C\')}\n    improved=sum(c<b for b,c in zip(drift[\'B\'],drift[\'C\']))\n    checks={\'reference_drift_positive\':median[\'B\']>0,\n            \'median_drift_reduced_30pct\':median[\'C\']<=RULES[\'drift_ratio_max\']*median[\'B\'],\n            \'improved_6_of_8\':improved>=RULES[\'improved_sources_min\']}\n    conditions={}\n    for shift in SHIFTS:\n        pos=[score(u,\'positive\',shift,\'C\') for u in IDS]\n        neg=[score(u,\'negative\',shift,\'C\') for u in IDS]\n        top1=sum(p[\'top1\'] for p in pos)\n        negative_top1=sum(n[\'top1\'] for n in neg)\n        pair=sum(p[\'registered\']>n[\'registered\'] for p,n in zip(pos,neg))\n        gap=statistics.median(p[\'gap\']/scales[\'C\'] for p in pos)\n        sid=geometry.shift_id(*shift)\n        conditions[sid]={\'positive_top1\':top1,\'negative_top1\':negative_top1,\n                         \'paired_positive_gt_negative\':pair,\'median_standardized_gap\':gap,\'denominator\':8}\n        checks[sid+\'/negative_top1_below_3\']=negative_top1<RULES[\'negative_top1_stop\']\n        if shift in SMALL:\n            checks[sid+\'/top1_6_of_8\']=top1>=RULES[\'shift_top1_min\']\n            checks[sid+\'/pair_6_of_8\']=pair>=RULES[\'shift_pair_min\']\n            checks[sid+\'/margin_retention_50pct\']=gap>=RULES[\'shift_margin_retention\']*zero_gate[\'median_standardized_gap\'][\'C\']\n    return {\'status\':\'PASSED\' if all(checks.values()) else \'FAILED\',\'checks\':checks,\n            \'worst_standardized_gap_change_by_source\':dict(zip(IDS,zip(drift[\'B\'],drift[\'C\']))),\n            \'median_worst_change\':median,\'improved_sources\':improved,\'conditions\':conditions}\n\n\ndef record_path(output,uid,role,shift,phase=False):\n    return output/(\'phase_records\' if phase else \'records\')/uid/role/(geometry.shift_id(*shift)+\'.json\')\n\n\ndef collect(output,phase=False):\n    result={}\n    for uid in IDS:\n        for role in ROLES:\n            for shift in SHIFTS[1:] if phase else SHIFTS:\n                path=record_path(output,uid,role,shift,phase)\n                if path.exists(): result[uid,role,*shift]=read(path)\n    return result\n\n\ndef summarize(output,status=\'INCOMPLETE\'):\n    actual,phase=collect(output),collect(output,True)\n    zero_gate=read(output/\'zero_gate.json\') if (output/\'zero_gate.json\').exists() else None\n    skipped=status in {\'STOPPED_ZERO_GATE\',\'STOPPED_UNINFORMATIVE_REFERENCE\'}\n    rows=[]\n    for uid in IDS:\n        for role in ROLES:\n            for shift in SHIFTS:\n                r=actual.get((uid,role,*shift),{\'status\':\'NOT_RUN_ZERO_GATE\' if skipped and shift!=(0,0) else \'PENDING\'})\n                rows.append({**r,\'unit_id\':uid,\'role\':role,\'dx_rgb\':shift[0],\'dy_rgb\':shift[1]})\n    counts={}\n    for role in ROLES:\n        for shift in SHIFTS:\n            selected=[r for r in rows if r[\'role\']==role and (r[\'dx_rgb\'],r[\'dy_rgb\'])==shift]\n            counts[role+\'/\'+geometry.shift_id(*shift)]={\'planned\':8,**{s:sum(r[\'status\']==s for r in selected)\n                for s in (\'SCORED\',\'PENDING\',\'NOT_RUN_ZERO_GATE\',\'OPERATIONAL_ERROR\')}}\n    summary={\'status\':status,\'planned_actual_records\':112,\'planned_phase_records\':96,\n             \'phase_scored\':sum(r[\'status\']==\'SCORED\' for r in phase.values()),\n             \'phase_not_run_zero_gate\':96 if skipped else 0,\'counts\':counts,\'zero_gate\':zero_gate,\n             \'shift_gate\':read(output/\'shift_gate.json\') if (output/\'shift_gate.json\').exists() else None,\n             \'records\':rows,\'claim_ceiling\':\'development_only_fixed_reader_no_FPR_no_blind_geometry_claim\'}\n    write(output/\'summary.json\',summary)\n    flat=[]\n    for r in rows:\n        for method in (\'B\',\'C\'):\n            row={k:r[k] for k in (\'unit_id\',\'role\',\'dx_rgb\',\'dy_rgb\',\'status\')}\n            row[\'reader\']=method\n            if r[\'status\']==\'SCORED\':\n                s=r[\'readers\'][method]\n                row.update({k:s[k] for k in (\'registered\',\'wrong_max\',\'gap\',\'top1\')})\n                row.update(s[\'scores\'])\n                if zero_gate: row[\'standardized_gap\']=s[\'gap\']/zero_gate[\'scales\'][method]\n            flat.append(row)\n    cols=list(dict.fromkeys(k for r in flat for k in r))\n    tmp=output/\'readout.csv.tmp\'\n    with tmp.open(\'w\',encoding=\'utf-8\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=cols);writer.writeheader();writer.writerows(flat)\n    os.replace(tmp,output/\'readout.csv\')\n    return summary\n\n\ndef save_attempt(output,uid,role,shift,compute,phase=False):\n    path=record_path(output,uid,role,shift,phase)\n    if path.exists() and read(path)[\'status\']==\'SCORED\': return read(path)\n    attempt=output/\'attempts\'/(\'phase\' if phase else \'actual\')/uid/role/geometry.shift_id(*shift)/(uuid.uuid4().hex+\'.json\')\n    write(attempt,{\'status\':\'STARTED\'})\n    try:\n        result={\'status\':\'SCORED\',**compute()}\n        write(attempt,result);write(path,result)\n        return result\n    except Exception as error:\n        result={\'status\':\'OPERATIONAL_ERROR\',**error_data(error)}\n        write(attempt,result);write(path,result);summarize(output)\n        raise\n\n\ndef run(source,output,repo):\n    import numpy as np\n    import torch\n    from cegwm.shared.keys import normalize_detection_key,public_key_digest\n    if output.resolve().is_relative_to(source.resolve()) or source.resolve().is_relative_to(output.resolve()):\n        raise ValueError(\'SOURCE and OUTPUT must not overlap\')\n    if sha(support.__file__)!=SUPPORT_SHA or sha(geometry.__file__)!=GEOMETRY_SHA:\n        raise ValueError(\'Validated support identity differs\')\n    protocol,old=inspect_source(source)\n    if subprocess.check_output([\'git\',\'-C\',str(repo),\'rev-parse\',\'HEAD\'],text=True).strip()!=EXACT:\n        raise ValueError(\'Historical source revision differs\')\n    if subprocess.check_output([\'git\',\'-C\',str(repo),\'diff\',\'HEAD\',\'--\'],text=True):\n        raise ValueError(\'Historical tracked source modified\')\n    if public_key_digest(normalize_detection_key(os.environ[\'CEG_WM_ROOT_KEY\']))!=protocol[\'key_digest\']:\n        raise ValueError(\'Key differs from source run\')\n    identity={\'schema\':\'crop_cross_spectrum_v1\',\'script_sha\':sha(__file__),\'support_sha\':SUPPORT_SHA,\n              \'geometry_sha\':GEOMETRY_SHA,\'source\':str(source.resolve()),\'source_exact\':EXACT,\n              \'source_hashes\':{n:sha(source/n) for n in (\'summary.json\',\'protocol.json\',\'environment.json\',\'model_revisions.json\')},\n              \'shifts_rgb\':[list(s) for s in SHIFTS],\'rules\':RULES,\n              \'candidate\':\'all_120_complex_offdiagonal_channel_pairs_per_frequency_energy_normalized_cosine\',\n              \'mask\':\'original_HF_common_no_Nyquist\',\'baseline\':\'original_common_mask_real_centered_cosine\',\n              \'scale\':\'zero_negative_within_image_16_wrong_key_population_RMS\',\n              \'no_black_subtraction\':True,\'no_search\':True}\n    bind(output/\'protocol.json\',identity)\n    # Scientific stop/completion is persistent: Run all cannot quietly reopen it.\n    if (output/\'summary.json\').exists() and read(output/\'summary.json\')[\'status\'] in TERMINAL:\n        print(\'Existing terminal result: \'+read(output/\'summary.json\')[\'status\'],flush=True)\n        return\n    if not torch.cuda.is_available(): raise RuntimeError(\'CUDA VAE reader required; no generation fallback\')\n    bind(output/\'environment.json\',{\'python\':platform.python_version(),\'gpu\':torch.cuda.get_device_name(0),\n        \'cuda\':torch.version.cuda,\'packages\':{p:version(p) for p in (\'torch\',\'torchvision\',\'diffusers\',\n            \'transformers\',\'numpy\',\'Pillow\',\'huggingface_hub\',\'safetensors\')}})\n    summarize(output)\n    reader=Reader(repo,source)\n    probe=FixedProbe(reader)\n    references={}\n    for uid in IDS:\n        for role in ROLES:\n            image=load_saved_image(source,old[uid,role,\'oracle_h\'][\'image\'])\n            z=reader.observe(image)\n            if tuple(z.shape)!=(1,16,64,64): raise ValueError(\'Unexpected VAE shape\')\n            production=reader.score(z)\n            check=replay_comparison(production,old[uid,role,\'oracle_h\'])\n            full=probe.linear.score(z.detach().cpu(),\'full\')\n            hf_ok=np.allclose(list(full[\'scores\'].values()),list(production[\'branches\'][\'hf\'].values()),atol=ATOL,rtol=RTOL)\n            if not check[\'passed\'] or not hf_ok:\n                write(output/\'replay_gate.json\',{\'status\':\'FAILED\',\'unit_id\':uid,\'role\':role,\'check\':check,\'full_HF_matches\':bool(hf_ok)})\n                raise RuntimeError(\'Historical score replay failed; no shifts allowed\')\n            scores=probe.score(z)\n            path=record_path(output,uid,role,(0,0))\n            if path.exists() and read(path)[\'status\']==\'SCORED\':\n                for method in (\'B\',\'C\'):\n                    if not np.allclose(list(scores[method][\'scores\'].values()),list(read(path)[\'readers\'][method][\'scores\'].values()),atol=ATOL,rtol=RTOL):\n                        raise RuntimeError(\'Saved zero-shift candidate score changed on resume\')\n            else:\n                write(path,{\'status\':\'SCORED\',\'readers\':scores,\'historical_replay\':check,\'production_reference\':production})\n            references[uid,role]=(image,z.detach().cpu())\n            print(f\'zero {uid[-4:]} {role}: B gap={scores["B"]["gap"]:.6f}, C gap={scores["C"]["gap"]:.6f}\',flush=True)\n    write(output/\'replay_gate.json\',{\'status\':\'PASSED\',\'planned\':16,\'atol\':ATOL,\'rtol\':RTOL})\n    zero={(u,r):read(record_path(output,u,r,(0,0))) for u in IDS for r in ROLES}\n    gate=assess_zero(zero)\n    bind(output/\'zero_gate.json\',gate)\n    print(\'ZERO GATE: \'+json.dumps(gate,ensure_ascii=False),flush=True)\n    if gate[\'status\']!=\'PASSED\':\n        status=\'STOPPED_ZERO_GATE\' if gate[\'status\']==\'FAILED\' else \'STOPPED_UNINFORMATIVE_REFERENCE\'\n        summarize(output,status)\n        print(status+\': no phase or RGB shift probes started.\',flush=True)\n        return\n    # Only after the scientific zero-shift gate: pure phase controls for all 17 keys.\n    for uid in IDS:\n        for role in ROLES:\n            _,z=references[uid,role]\n            for shift in SHIFTS[1:]:\n                def phase_compute():\n                    scores=probe.score(robust_phase_shift(z,*shift))\n                    diff=max(abs(scores[\'C\'][\'scores\'][k]-zero[uid,role][\'readers\'][\'C\'][\'scores\'][k]) for k in LABELS)\n                    if diff>RULES[\'phase_score_atol\']:\n                        raise RuntimeError(f\'Pure phase invariance failed: {diff}\')\n                    return {\'readers\':scores,\'max_abs_C_score_change\':diff}\n                save_attempt(output,uid,role,shift,phase_compute,True)\n    phase=collect(output,True)\n    write(output/\'phase_gate.json\',{\'status\':\'PASSED\',\'planned\':96,\n          \'max_abs_C_score_change\':max(r[\'max_abs_C_score_change\'] for r in phase.values())})\n    for shift in SHIFTS[1:]:\n        for uid in IDS:\n            for role in ROLES:\n                image,_=references[uid,role]\n                def actual_compute():\n                    zero_rgb=geometry.rgb_shift(image,*shift,False)\n                    circular=geometry.rgb_shift(image,*shift,True)\n                    if not np.array_equal(np.asarray(zero_rgb),np.asarray(circular)):\n                        raise RuntimeError(\'RGB boundary controls differ; stop, do not select favorable boundary\')\n                    return {\'readers\':probe.score(reader.observe(zero_rgb)),\n                            \'rgb_boundary_identical\':True,\'rgb_interpolation\':\'none_integer_copy\'}\n                save_attempt(output,uid,role,shift,actual_compute)\n        summarize(output)\n        print(\'Completed actual shift \'+geometry.shift_id(*shift),flush=True)\n    final_gate=assess_shifts(collect(output),gate)\n    write(output/\'shift_gate.json\',final_gate)\n    status=\'COMPLETE_CANDIDATE_PASSED\' if final_gate[\'status\']==\'PASSED\' else \'COMPLETE_CANDIDATE_REJECTED\'\n    summarize(output,status)\n    print(status+\'\\n\'+json.dumps(final_gate,ensure_ascii=False,indent=2),flush=True)\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    for name in (\'source\',\'output\',\'repo\'): parser.add_argument(\'--\'+name,type=Path,required=True)\n    args=parser.parse_args()\n    if args.output.resolve().is_relative_to(args.source.resolve()) or args.source.resolve().is_relative_to(args.output.resolve()):\n        raise ValueError(\'SOURCE and OUTPUT must not overlap\')\n    sys.path[:0]=[str(args.repo),str(args.repo/\'src\')]\n    try: run(args.source,args.output,args.repo)\n    except Exception as error:\n        write(args.output/\'last_failure.json\',error_data(error))\n        summarize(args.output,\'OPERATIONAL_ERROR\')\n        raise\n\n\nif __name__==\'__main__\': main()\n', encoding='utf-8')


## 冻结的先后顺序与判据

1. 检查全部源图像哈希、原代码/模型/密钥；重放 16 个 Oracle 的原 LF/HF/联合分数。工程错误单独记录。
2. 只读取 16 个零位移样本，计算 B/C 的全部 17 钥分数。
3. **零位移门槛**：C 正样本注册钥第一名 ≥7/8；配对正样本注册钥分数高于负样本 ≥7/8；
   C 标准化间隔中位数 >0 且 ≥B 的 80%；负样本注册钥第一名 <3/8。
   B 中位数非正时记为参照不足并停止，不把无法比较当作 C 通过。
4. 只有上述门槛通过，才检查纯潜变量相位平移下 C 全部钥分数变化 ≤1e-6。
5. 通过数学检查后，仅执行 x/y 各 ±2、x/y 各 +8 RGB 像素。整数数组复制，不插值。
   黑填充和循环移位若不相同，停止而非挑选有利边界。
6. **位移门槛**：四个 ±2 条件分别要求注册钥第一名 ≥6/8、正分数高于配对负样本 ≥6/8、
   C 间隔中位数保留零位移的 ≥50%。最坏间隔变化的中位数比 B 降低 ≥30%，至少 6/8 图像改善。
   所有条件的负样本第一名均须 <3/8。8 像素仅作整格点参照，不代替 ±2 像素成功。

间隔 = 注册钥分数 − 固定 16 错钥最大分数。B/C 各自的报告尺度固定为：8 个零位移负样本中，
16 错钥分数减去本图错钥均值后的平方偏差，汇总取均值再开方（总体 RMS，不做无偏修正）。
该尺度仅标准化报告，不改变读法或密钥排名，也不是检测阈值。零/非有限尺度为工程不可用。
平局不计为第一名，正负同分不计为配对胜出；原始分数全部保留。

这些是开发筛选门槛，不是置信度或 FPR 保证。图像及其重复位移不是独立扩充样本。


In [ ]:
run_logged([sys.executable, str(script), '--source', str(SOURCE), '--output', str(output),
            '--repo', str(checkout)], 'cross-spectrum-diagnostic', env=child_env, cwd=checkout)
summary = json.loads((output / 'summary.json').read_text())
print('STATUS:', summary['status'])
print('ZERO GATE:', json.dumps(summary['zero_gate'], ensure_ascii=False, indent=2))
if summary['status'].startswith('STOPPED_'):
    print('已按预定门槛停止：没有运行纯相位或实际位移对照，不自动调参重试。')
elif summary['shift_gate'] is not None:
    print('SHIFT GATE:', json.dumps(summary['shift_gate'], ensure_ascii=False, indent=2))
print('Saved result directory:', output)


## 结果检查

零位移图始终显示 B/C 的逐图标准化密钥间隔。只有完整位移对照结束，才显示位移曲线。
图中的零线仅表示注册钥与最强错钥打平，**不是经过校准的水印存在阈值**。
位移图中每条细线是一张图像，同一图像重复观测不增加独立样本数。


In [ ]:
import matplotlib.pyplot as plt
rows=summary['records']
gate=summary['zero_gate']
if gate is not None:
    scales=gate['scales']
    fig,axes=plt.subplots(1,2,figsize=(12,4),sharey=True,layout='constrained')
    for ax,role in zip(axes,('positive','negative')):
        zero=sorted([r for r in rows if r['role']==role and r['dx_rgb']==0 and r['dy_rgb']==0],key=lambda r:r['unit_id'])
        for method,marker,color,offset in [('B','o','#0072B2',-.08),('C','^','#D55E00',.08)]:
            ax.scatter([i+1+offset for i in range(8)],
                       [r['readers'][method]['gap']/scales[method] for r in zero],
                       marker=marker,color=color,label=method)
        ax.axhline(0,color='gray',linewidth=.8)
        ax.set(title=f'Zero shift: {role} (8 sources)',xlabel='Source index',xticks=range(1,9))
        ax.grid(axis='y',alpha=.2)
        ax.legend()
    axes[0].set_ylabel('Key gap / frozen wrong-key RMS')
    fig.suptitle(summary['status'])
    fig.savefig(output/'zero_key_gaps.png',dpi=150)
    plt.show()
    if summary['status'].startswith('COMPLETE_'):
        shifts=[(0,0),(-2,0),(2,0),(0,-2),(0,2),(8,0),(0,8)]
        labels=['0','x-2','x+2','y-2','y+2','x+8','y+8']
        fig,axes=plt.subplots(2,2,figsize=(12,7),sharex=True,sharey=True,layout='constrained')
        for i,role in enumerate(('positive','negative')):
            for j,method in enumerate(('B','C')):
                ax=axes[i,j]
                for uid in sorted({r['unit_id'] for r in rows}):
                    data={(r['dx_rgb'],r['dy_rgb']):r for r in rows if r['unit_id']==uid and r['role']==role}
                    ax.plot(range(7),[data[s]['readers'][method]['gap']/scales[method] for s in shifts],marker='.',alpha=.6)
                ax.axhline(0,color='gray',linewidth=.8)
                ax.set_title(f'{role}, reader {method}, 8 sources')
                ax.set_xticks(range(7),labels)
                ax.grid(axis='y',alpha=.2)
                if j==0: ax.set_ylabel('Standardized key gap')
                if i==1: ax.set_xlabel('Fixed RGB shift (pixels)')
        fig.savefig(output/'shift_key_gaps.png',dpi=150)
        plt.show()


## 解释与恢复

- `STOPPED_ZERO_GATE`：零位移已不能满足保留密钥证据的要求，本候选停止，96 个实际位移与 96 个纯相位观测均标为未运行。
- `STOPPED_UNINFORMATIVE_REFERENCE`：参照基线不足以支持预定比例比较，不等于候选通过。
- `COMPLETE_CANDIDATE_REJECTED`：零位移通过但完整固定对照未满足全部门槛；不增加搜索补救。
- `COMPLETE_CANDIDATE_PASSED`：仅支持已有 8 对图像上的开发假设，不是独立验证、低 FPR 检测或完整盲几何恢复。
- `OPERATIONAL_ERROR`：工程失败，不作为科学否定；查看 `last_failure.json` 和 `execution_logs/`。

中断后沿用 SOURCE/RUN_ID 运行全部：已完成记录跳过，未终结任务的基线会重放；终结的科学结果直接返回，
不会重新打开候选。协议/代码/源文件/环境不一致会停止。不要同时写同一输出目录。
输出包含 `summary.json`、`readout.csv`、`zero_gate.json`、`replay_gate.json`、
通过后才有的 `phase_gate.json`/`shift_gate.json`、`records/`、`phase_records/`、`attempts/` 和日志。
数据缺失不会触发重新生成图像。
